In [1]:
from data import *
import pandas as pd

In [2]:
P = get_data_paths()
train_data, test_data, sub_data = get_train_test()

train_csv found at kaggle\input\neurips-open-polymer-prediction-2025\train.csv [OK]
test_csv found at kaggle\input\neurips-open-polymer-prediction-2025\test.csv [OK]
sample_submission found at kaggle\input\neurips-open-polymer-prediction-2025\sample_submission.csv [OK]
tc_smiles found at kaggle\input\tc-smiles\Tc_SMILES.csv [OK]
sed_bigsmiles found at kaggle\input\smiles-extra-data\JCIM_sup_bigsmiles.csv [OK]
sed_tg3 found at kaggle\input\smiles-extra-data\data_tg3.xlsx [OK]
sed_dnst1 found at kaggle\input\smiles-extra-data\data_dnst1.xlsx [OK]
dataset4 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset4.csv [OK]
dataset1 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset1.csv [OK]
dataset2 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset2.csv [OK]
dataset3 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset3.csv [OK]


In [3]:
train_data.isna().sum()

id            0
SMILES        0
Tg         7462
FFV         943
Tc         7236
Density    7360
Rg         7359
dtype: int64

In [4]:
df_extra = pd.read_csv("kaggle/input/tc-smiles/Tc_SMILES.csv")
target="TC_mean"
df_extra = df_extra[['SMILES', target]].dropna(subset=['SMILES']).copy()
print(df_extra.isna().sum())

SMILES     0
TC_mean    0
dtype: int64


In [5]:
A = pd.DataFrame({'SMILES': ['A', 'A', 'B'], 'TC_mean': [1.0, 3.0, 2.0]})
B = pd.DataFrame({'SMILES': ['A', 'C', 'C'], 'TC_mean': [1.5, 2.5, 3.2]})


In [6]:
train = combine_data(train_data, P['tc_smiles'], target='Tc', source_name='TC_mean')

------------------------------------------------------------
[START] Adding extra data from 'C:\Users\Teyer\my_wdmpnn\kaggle\input\tc-smiles\Tc_SMILES.csv' targeting 'TC_mean' to train target 'Tc'.
[INFO] Train data shape: (7973, 7)
[INFO] Extra data shape: (874, 2), columns: ['Tc', 'SMILES']
[INFO] Found 737 overlapping SMILES.
[INFO] Combined data shape: (8103, 7)


In [7]:
# utils: try variants to make RDKit accept a SMILES
from rdkit import Chem
from rdkit.Chem import rdchem
import re
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def try_mol_from_smiles(smi, sanitize=True):
    try:
        m = Chem.MolFromSmiles(smi, sanitize=sanitize)
        return m
    except Exception:
        return None

def sanitize_mol_safe(m):
    try:
        Chem.SanitizeMol(m)
        return True
    except Exception:
        try:
            # 尝试部分 sanitize（跳过属性计算类的检查）
            Chem.SanitizeMol(m, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES)
            return True
        except Exception:
            return False

def pick_largest_fragment_smiles(smi):
    frags = smi.split('.')
    # 返回按长度（或原子数）排第一个可解析的片段
    frags_sorted = sorted(frags, key=lambda s: len(s), reverse=True)
    return frags_sorted

def replace_R_groups(smi, repl='C'):
    # 把 [R], [R'], [R1]... 等替换为 repl（默认 C）
    s = re.sub(r'\[R[^\]]*\]', repl, smi)
    return s

def remove_R_groups(smi):
    return re.sub(r'\[R[^\]]*\]', '', smi)

def replace_asterisk(smi, repl='C'):
    return smi.replace('*', repl)

def remove_asterisk(smi):
    return smi.replace('*','')

def clean_extra_spaces(smi):
    return re.sub(r'\s+', '', smi)

In [8]:
def canonicalize_try_all(smi):
    smi = smi.strip()
    attempts = []
    # 1) 原始尝试
    m = try_mol_from_smiles(smi, sanitize=True)
    if m:
        return ('original', Chem.MolToSmiles(m, canonical=True))
    attempts.append('original_failed')

    # 2) sanitize=False then try sanitize manually
    m = try_mol_from_smiles(smi, sanitize=False)
    if m and sanitize_mol_safe(m):
        return ('sanitize_false_then_sanitize', Chem.MolToSmiles(m, canonical=True))
    attempts.append('sanitize_false_failed')

    # 3) try largest fragment only
    for frag in pick_largest_fragment_smiles(smi):
        m = try_mol_from_smiles(frag, sanitize=True)
        if m:
            return ('largest_fragment', Chem.MolToSmiles(m, canonical=True))

    # 4) replace bracketed R-groups with 'C'
    s = replace_R_groups(smi, repl='C')
    m = try_mol_from_smiles(s, sanitize=True)
    if m:
        return ('R->C', Chem.MolToSmiles(m, canonical=True))

    # 5) replace bracketed R-groups with '*' (wildcard)
    s = replace_R_groups(smi, repl='*')
    m = try_mol_from_smiles(s, sanitize=True)
    if m:
        return ("R->*", Chem.MolToSmiles(m, canonical=True))

    # 6) remove bracketed R-groups entirely
    s = remove_R_groups(smi)
    m = try_mol_from_smiles(s, sanitize=True)
    if m:
        return ('R_removed', Chem.MolToSmiles(m, canonical=True))

    # 7) replace '*' with 'C'
    s = replace_asterisk(smi, repl='C')
    m = try_mol_from_smiles(s, sanitize=True)
    if m:
        return ('*->C', Chem.MolToSmiles(m, canonical=True))

    # 8) remove '*' completely
    s = remove_asterisk(smi)
    m = try_mol_from_smiles(s, sanitize=True)
    if m:
        return ('* removed', Chem.MolToSmiles(m, canonical=True))

    # 9) combos: R->C then remove asterisk, try fragments
    s = replace_R_groups(smi, 'C')
    s = replace_asterisk(s, 'C')
    for frag in pick_largest_fragment_smiles(s):
        m = try_mol_from_smiles(frag, sanitize=True)
        if m:
            return ('R->C & *->C & fragment', Chem.MolToSmiles(m, canonical=True))

    return (None, None)

In [9]:
import pandas as pd

# 你的示例
df = pd.DataFrame({
    "SMILES": [
        "*C(F)(F)CC(F)([R])C(*)(F)F",
        "*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O",
        "*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4",
        "*OC2OC(CO[R])C(OC1OC(CO[R])C(*)C(O[R])C1O[R])C(O[R])C2O[R]",
        "*O[Si](*)([R])[R]",
        "O=C=N[R1]N=C=O.O[R2]O.O[R3]O"
    ]
})

results = []
for smi in df['SMILES']:
    method, can = canonicalize_try_all(smi)
    results.append({'SMILES':smi, 'method':method, 'canonical':can})
res = pd.DataFrame(results)
print(res)

                                              SMILES method  \
0                         *C(F)(F)CC(F)([R])C(*)(F)F   R->C   
1  *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)c...   R->C   
2  *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c...   R->C   
3  *OC2OC(CO[R])C(OC1OC(CO[R])C(*)C(O[R])C1O[R])C...   R->C   
4                                  *O[Si](*)([R])[R]   R->C   
5                       O=C=N[R1]N=C=O.O[R2]O.O[R3]O   R->C   

                                           canonical  
0                           *C(F)(F)CC(C)(F)C(*)(F)F  
1      *CN(C)Cc1cc(Cc2cc(*)c(O)c(CN(C)C*)c2)cc(*)c1O  
2  *NC(=O)c1ccc2c(c1)C(=O)N(c1ccc(Cc3ccc(*)cc3)cc...  
3     *OC1OC(COC)C(OC2OC(COC)C(*)C(OC)C2OC)C(OC)C1OC  
4                                      *O[Si](*)(C)C  
5                                O=C=NCN=C=O.OCO.OCO  


In [10]:
def try_star_replacements(smi):
    # 先把 bracketed R 处理成 C（如果尚未做）
    s1 = replace_R_groups(smi, 'C')
    # 1) 尝试把 '*' -> 'C'
    s2 = replace_asterisk(s1, repl='C')
    m = try_mol_from_smiles(s2, sanitize=True)
    if m and sanitize_mol_safe(m):
        return ('R->C & *->C', Chem.MolToSmiles(m, canonical=True))
    # 2) 再尝试移除 '*'（更激进）
    s3 = remove_asterisk(s1)
    m2 = try_mol_from_smiles(s3, sanitize=True)
    if m2 and sanitize_mol_safe(m2):
        return ('R->C & * removed', Chem.MolToSmiles(m2, canonical=True))
    return (None, None)

# 在 DataFrame 上批量尝试
results = []
for smi in df['SMILES']:
    method, can = canonicalize_try_all(smi)   # 你已有的函数
    if method is None:
        method2, can2 = try_star_replacements(smi)
        method, can = method2 or method, can2 or can
    results.append({'SMILES': smi, 'method': method, 'canonical': can})
res = pd.DataFrame(results)
print(res)

                                              SMILES method  \
0                         *C(F)(F)CC(F)([R])C(*)(F)F   R->C   
1  *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)c...   R->C   
2  *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c...   R->C   
3  *OC2OC(CO[R])C(OC1OC(CO[R])C(*)C(O[R])C1O[R])C...   R->C   
4                                  *O[Si](*)([R])[R]   R->C   
5                       O=C=N[R1]N=C=O.O[R2]O.O[R3]O   R->C   

                                           canonical  
0                           *C(F)(F)CC(C)(F)C(*)(F)F  
1      *CN(C)Cc1cc(Cc2cc(*)c(O)c(CN(C)C*)c2)cc(*)c1O  
2  *NC(=O)c1ccc2c(c1)C(=O)N(c1ccc(Cc3ccc(*)cc3)cc...  
3     *OC1OC(COC)C(OC2OC(COC)C(*)C(OC)C2OC)C(OC)C1OC  
4                                      *O[Si](*)(C)C  
5                                O=C=NCN=C=O.OCO.OCO  


In [11]:
# 1) 让 pandas 显示完整列内容
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(res)                     # 现在 canonical 列不会被截断

# 2) 把整个 DataFrame 打成字符串（不省略）
print(res.to_string(index=False))

# 3) 按行打印具体字段，保证完整显示
for i, row in res.iterrows():
    print(i, row['method'])
    print(row['canonical'])
    print('-'*80)

# 4) 如果想检查 canonical 是否仍含 '*' 或是否能被 RDKit 解析
from rdkit import Chem
for s in res['canonical']:
    print('*' in str(s), Chem.MolFromSmiles(str(s)) is not None)

                                                       SMILES method  \
0                                  *C(F)(F)CC(F)([R])C(*)(F)F   R->C   
1       *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O   R->C   
2      *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4   R->C   
3  *OC2OC(CO[R])C(OC1OC(CO[R])C(*)C(O[R])C1O[R])C(O[R])C2O[R]   R->C   
4                                           *O[Si](*)([R])[R]   R->C   
5                                O=C=N[R1]N=C=O.O[R2]O.O[R3]O   R->C   

                                              canonical  
0                              *C(F)(F)CC(C)(F)C(*)(F)F  
1         *CN(C)Cc1cc(Cc2cc(*)c(O)c(CN(C)C*)c2)cc(*)c1O  
2  *NC(=O)c1ccc2c(c1)C(=O)N(c1ccc(Cc3ccc(*)cc3)cc1)C2=O  
3        *OC1OC(COC)C(OC2OC(COC)C(*)C(OC)C2OC)C(OC)C1OC  
4                                         *O[Si](*)(C)C  
5                                   O=C=NCN=C=O.OCO.OCO  
                                                    SMILES method                        

In [12]:
train, test, sub = get_train_test()
train_added = add_extra_data(train)
train_cleaned = clean_smiles(train_added)
train_filtered = filter_train_data(train_cleaned)

train_csv found at kaggle\input\neurips-open-polymer-prediction-2025\train.csv [OK]
test_csv found at kaggle\input\neurips-open-polymer-prediction-2025\test.csv [OK]
sample_submission found at kaggle\input\neurips-open-polymer-prediction-2025\sample_submission.csv [OK]
tc_smiles found at kaggle\input\tc-smiles\Tc_SMILES.csv [OK]
sed_bigsmiles found at kaggle\input\smiles-extra-data\JCIM_sup_bigsmiles.csv [OK]
sed_tg3 found at kaggle\input\smiles-extra-data\data_tg3.xlsx [OK]
sed_dnst1 found at kaggle\input\smiles-extra-data\data_dnst1.xlsx [OK]
dataset4 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset4.csv [OK]
dataset1 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset1.csv [OK]
dataset2 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset2.csv [OK]
dataset3 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset3.csv [OK]
-------------------------------------

In [23]:
# utils: count SMILES containing '*' or bracketed R-groups
import re
import pandas as pd

def count_placeholder_smiles(df: pd.DataFrame, col: str = "SMILES", sample_n: int = 5) -> dict:
    """
    Count rows containing '*' and any uppercase 'R' anywhere (bracketed or not).
    Returns counts, percentages, example rows and boolean masks.
    """
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in DataFrame.")
    s = df[col].astype(str)

    # 星号 '*' 任意位置
    star_mask = s.str.contains(r"\*", na=False)
    # 任意位置的大写 R（包括 [R], [R1], 也包括独立的 R）
    any_R_mask = s.str.contains(r"R", na=False)

    total = len(df)
    star_count = int(star_mask.sum())
    any_R_count = int(any_R_mask.sum())
    either_mask = star_mask | any_R_mask
    either_count = int(either_mask.sum())

    return {
        "total_rows": total,
        "star_count": star_count,
        "star_pct": star_count / total if total else 0.0,
        "any_R_count": any_R_count,
        "any_R_pct": any_R_count / total if total else 0.0,
        "either_count": either_count,
        "either_pct": either_count / total if total else 0.0,
        "examples_star": df.loc[star_mask].head(sample_n).copy(),
        "examples_any_R": df.loc[any_R_mask].head(sample_n).copy(),
        "examples_either": df.loc[either_mask].head(sample_n).copy(),
        "masks": {
            "star_mask": star_mask,
            "any_R_mask": any_R_mask,
            "either_mask": either_mask
        }
    }


In [24]:
stats = count_placeholder_smiles(train_added, col="SMILES")
print(stats["star_count"], stats["any_R_count"], stats["either_count"])
display(stats["examples_either"])

10137 6 10138


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817.0,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919.0,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5ccc(N*)cc5)cc4)CCC(CCCCC)CC3)cc2)cc1,NaN,0.370410,NaN,NaN,NaN
2,388772.0,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(S(=O)(=O)c7ccc(Oc8ccc(C=C9CCCC(=Cc%10ccc(*)cc%10)C9=O)cc8)cc7)cc6)cc5)CCCCC4)cc3)cc2)cc1,NaN,0.378860,NaN,NaN,NaN
3,519416.0,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)c(-c3ccc(C)cc3)c2-c2ccc(C)cc2)cc1,NaN,0.387324,NaN,NaN,NaN
4,539187.0,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N+](=O)[O-])cc3)c(C(*)=O)cc2OCCCCCCCCCOCC2CCCN2c2ccc([N+](=O)[O-])cc2)cc1,NaN,0.355470,NaN,NaN,NaN
